# Privacy-Preserving Federated Learning via Secret Sharing and Multi-Key Homomorphic Encryption (CDKS-LSS)

**Complete Academic & Research Implementation of:**
> **"Privacy-preserving federated learning via secret sharing and multi-key homomorphic encryption"**  
> *Yuntao Wang, Fumiya Inoue, Yujie Gu, Xun Shen, and Mingwu Zhang*  
> *Information Sciences*, 2026.

---

## Executive Summary & Architecture

This notebook aggregates the complete research codebase of the **CDKS-LSS** privacy-preserving federated learning framework.

### Core Problem
In standard Federated Learning (FedAvg), clients send model weights or gradients in plaintext. The central server can perform **gradient inversion attacks** to reconstruct private local training data. While Multi-Key Homomorphic Encryption (MK-HE) like the **CDKS** scheme enables encrypted aggregation with individual client keys, standard CDKS suffers from a critical vulnerability:
$$\mu_i = c_{i,1} \cdot s_i + e_i^* \implies c_{i,0} + \mu_i \approx m_i$$
When client $i$ transmits their partial decryption value $\mu_i$ to the server, the server can trivially recover the client's individual plaintext model update $m_i$!

### Proposed CDKS-LSS Solution
The paper introduces **CDKS-LSS**, which resolves this leakage by having participants **secret-share** their partial decryption values $\mu_i$ among themselves using Shamir's Linear Secret Sharing (LSS) over the polynomial ring $R_q$:
1. Participant $i$ computes $\mu_i = c_{i,1} \cdot s_i$.
2. Participant $i$ generates a random polynomial $f_i(X)$ over $R_q$ of degree $t-1$ such that $f_i(0) = \mu_i$.
3. Participants exchange evaluations $f_i(\alpha_j)$ with peer $j$.
4. Each participant $j$ locally aggregates their received shares: $\tilde{s}_j = \sum_{i=1}^N f_i(\alpha_j)$.
5. Participants send only the aggregated shares $\tilde{s}_j$ to the server.
6. The server collects any $t$ out of $N$ aggregated shares and applies Lagrange interpolation over $R_q$ to reconstruct $F(0) = \sum_{i=1}^N \mu_i$.
7. The server decrypts the aggregate model update: $M = c_0 + \sum \mu_i \approx \sum m_i$.

**Result:** The server reconstructs the global aggregate while learning **zero information** about individual updates $m_i$, and tolerates up to $N - t$ client dropouts!

```text
========================================================================================
                                CDKS-LSS PROTOCOL FLOW
========================================================================================
[Server]                                                      [Clients 1 ... N]
   |                                                                  |
   | ------------ 1. Broadcast global weights w_G ------------------> |
   |                                                                  | (Local SGD on D_i)
   |                                                                  | w_i = LocalTrain(w_G)
   |                                                                  |
   |                                                                  | Encrypt with own pk_i:
   |                                                                  | (c_{i,0}, c_{i,1}) = Enc(pk_i, w_i)
   | <----------- 2. Submit c_{i,0} first components ---------------- |
   |                                                                  |
   | (Homomorphic addition: c_0 = sum c_{i,0})                        |
   |                                                                  | Compute mu_i = c_{i,1} * s_i
   |                                                                  | Create f_i(X) with f_i(0)=mu_i
   |                                                                  | Exchange f_i(alpha_j) (peer-to-peer)
   |                                                                  | Locally aggregate: s_tilde_j = sum f_i(alpha_j)
   |                                                                  |
   | <----------- 3. Submit aggregated shares s_tilde_j -------------- |
   |                                                                  |
   | (Select t shares; Lagrange interpolation -> sum mu_i)
   | (Reconstruct M = c_0 + sum mu_i approx sum w_i)
   | (Update w_G = M / N)
========================================================================================
```

---

## Table of Contents
1. [Environment Setup & Global Configuration](#0-environment-setup--global-configuration)
2. [Module 1: Polynomial Ring Arithmetic ($R_q = \mathbb{Z}_q[X]/(X^n+1)$)](#1-polynomial-ring-arithmetic)
3. [Module 2: Ring Learning With Errors (RLWE)](#2-ring-learning-with-errors-rlwe)
4. [Module 3: CDKS Multi-Key Homomorphic Encryption](#3-cdks-multi-key-homomorphic-encryption)
5. [Module 4: Shamir Linear Secret Sharing (Classic & Ring) & Re-Sharing](#4-shamir-linear-secret-sharing)
6. [Module 5: The Proposed CDKS-LSS Scheme](#5-the-proposed-cdks-lss-scheme)
7. [Module 6: Baseline xMK-CKKS Scheme](#6-baseline-xmk-ckks-scheme)
8. [Module 7: Security Vulnerability & Attack Demonstrations](#7-security-vulnerability--attack-demonstrations)
9. [Module 8: Machine Learning Model (Logistic Regression)](#8-machine-learning-model)
10. [Module 9: Synthetic Dataset Generation & Non-IID Partitioning](#9-synthetic-dataset-generation)
11. [Module 10: Utility Functions & Communication Overhead Metrics](#10-utility-functions--metrics)
12. [Module 11: Federated Learning Entities (Client & Server)](#11-federated-learning-entities)
13. [Module 12: Federated Learning Orchestration Loops](#12-federated-learning-orchestration)
14. [Module 13: Comprehensive Unit Test Suite](#13-comprehensive-unit-test-suite)
15. [Experiment 1: Plain FedAvg Baseline](#14-experiment-1-plain-fedavg-baseline)
16. [Experiment 2: CDKS Encrypted FL + Vulnerability Demonstration](#15-experiment-2-cdks-encrypted-fl)
17. [Experiment 3: CDKS-LSS Privacy-Preserving FL (Proposed Method)](#16-experiment-3-cdks-lss-privacy-preserving-fl)
18. [Experiment 4: xMK-CKKS Baseline Comparison](#17-experiment-4-xmk-ckks-baseline-comparison)
19. [Experiment 5: Security Attacks Comprehensive Evaluation](#18-experiment-5-security-demonstration)
20. [Experiment 6: Client Dropout Tolerance Evaluation](#19-experiment-6-client-dropout-tolerance)
21. [Experiment 7: Communication Overhead Analysis](#20-experiment-7-communication-overhead-analysis)
22. [Master Benchmark Runner](#21-master-benchmark-runner)
23. [Thesis & Viva Defense Guide](#22-thesis--viva-defense-guide)

---
> ⚠️ **Educational Implementation Note:** This notebook uses educational parameters ($n=64, q=1048583$) so computations run rapidly and coefficients remain traceable. A real-world production deployment requires cryptographic parameter sizes ($n \ge 4096, q \sim 2^{100+}$).


## 0. Environment Setup & Global Configuration

We initialize all libraries, configure `%matplotlib inline` with headless fallback, and define global parameters.


In [ ]:
# Environment Setup
import os
import sys
import time
import math
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib

# Enable inline plotting in Jupyter/IPython, or fallback to Agg for headless environments
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

np.set_printoptions(linewidth=120, precision=4)

# ============================================================================
# 1. RING PARAMETERS (Section 2.2)
# ============================================================================
# Ring R_q = Z_q[X] / (X^n + 1)
# n : polynomial degree (power of 2 for cyclotomic polynomial)
# q : ciphertext modulus (prime)
RING_N = 64                    # Polynomial degree
RING_Q = 1048583               # Ciphertext modulus (prime near 2^20)

# ============================================================================
# 2. NOISE / SAMPLING PARAMETERS (Section 2.2)
# ============================================================================
ERROR_BOUND = 3                # Coefficients of error polynomials in [-3, 3]
SMUDGE_BOUND = 50              # Smudging noise bound for partial decryption (phi)
DELTA = 1                      # Plaintext scaling factor

# ============================================================================
# 3. FEDERATED LEARNING & ML PARAMETERS
# ============================================================================
NUM_CLIENTS = 5                # N = number of participants
THRESHOLD = 3                  # t = LSS reconstruction threshold (t <= N)
NUM_FEATURES = 20              # Dimensionality of feature space
NUM_SAMPLES_PER_CLIENT = 100   # Local dataset size per client
NUM_CLASSES = 2                # Binary classification
LEARNING_RATE = 0.1            # SGD learning rate
LOCAL_EPOCHS = 5               # Local training epochs per FL round
FL_ROUNDS = 20                 # Number of federated learning rounds
RANDOM_SEED = 42               # For reproducibility
WEIGHT_SCALE = 1000            # Quantization scale for float -> int encoding

# ============================================================================
# 4. SHAMIR LSS PARAMETERS (Section 2.5 & 2.6)
# ============================================================================
SHAMIR_PRIME = 104729          # Prime for classic Shamir demo over Z_p

def get_evaluation_points(n_clients: int) -> List[int]:
    """
    Generate evaluation points alpha_1, ..., alpha_N for Shamir LSS.
    Uses 1, 2, ..., n_clients. Since q is prime and |alpha_i - alpha_j| < q,
    every difference is invertible in Z_q, fulfilling the Exceptional Sequence condition.
    """
    return list(range(1, n_clients + 1))

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Configuration loaded: Ring degree n={RING_N}, Modulus q={RING_Q}, Clients N={NUM_CLIENTS}, Threshold t={THRESHOLD}")


## 1. Polynomial Ring Arithmetic ($R_q = \mathbb{Z}_q[X]/(X^n+1)$)

### Mathematical Formulation
The CDKS-LSS framework operates on the quotient polynomial ring:
$$R = \mathbb{Z}[X] / (X^n + 1)$$
$$R_q = R / qR = \mathbb{Z}_q[X] / (X^n + 1)$$

Elements are polynomials of degree $< n$:
$$a(X) = a_0 + a_1 X + a_2 X^2 + \dots + a_{n-1} X^{n-1}, \quad a_i \in \left(-\frac{q}{2}, \frac{q}{2}\right]$$

### Negacyclic Reduction
In the quotient ring, $X^n + 1 \equiv 0 \implies X^n \equiv -1$. When two polynomials of degree $n-1$ are multiplied:
$$X^{n+k} = X^k \cdot X^n \equiv -X^k$$
High-degree coefficients wrap around to low degrees with a **sign flip** (negacyclic convolution).


In [ ]:
# ============================================================================
# POLYNOMIAL RING ARITHMETIC (R_q = Z_q[X] / (X^n + 1))
# ============================================================================

def poly_mod_q(poly: np.ndarray, q: int) -> np.ndarray:
    """Reduce polynomial coefficients into the symmetric centered range (-q/2, q/2]."""
    result = np.array(poly, dtype=np.int64) % q
    result = np.where(result > q // 2, result - q, result)
    return result

def poly_add(a: np.ndarray, b: np.ndarray, q: int) -> np.ndarray:
    """Add two polynomials in R_q: (a + b) mod q."""
    return poly_mod_q(a + b, q)

def poly_sub(a: np.ndarray, b: np.ndarray, q: int) -> np.ndarray:
    """Subtract two polynomials in R_q: (a - b) mod q."""
    return poly_mod_q(a - b, q)

def poly_mul(a: np.ndarray, b: np.ndarray, q: int) -> np.ndarray:
    """
    Multiply two polynomials in R_q = Z_q[X] / (X^n + 1).
    Uses standard convolution followed by negacyclic reduction modulo X^n + 1.
    """
    n = len(a)
    full_product = np.convolve(a.astype(np.int64), b.astype(np.int64))
    result = np.zeros(n, dtype=np.int64)
    result[:] = full_product[:n]
    if len(full_product) > n:
        high = full_product[n:]
        result[:len(high)] -= high  # X^n = -1 wrap-around
    return poly_mod_q(result, q)

def poly_scalar_mul(poly: np.ndarray, scalar: int, q: int) -> np.ndarray:
    """Multiply a polynomial by an integer scalar in R_q: (scalar * poly) mod q."""
    return poly_mod_q(poly * scalar, q)

def poly_negate(poly: np.ndarray, q: int) -> np.ndarray:
    """Negate a polynomial in R_q: (-poly) mod q."""
    return poly_mod_q(-poly, q)

def sample_uniform(n: int, q: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Sample a uniformly random polynomial from R_q (a <- R_q)."""
    if rng is None:
        rng = np.random.default_rng()
    coeffs = rng.integers(0, q, size=n, dtype=np.int64)
    return poly_mod_q(coeffs, q)

def sample_ternary(n: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Sample ternary polynomial coefficients from {-1, 0, 1} (s <- chi)."""
    if rng is None:
        rng = np.random.default_rng()
    return rng.integers(-1, 2, size=n, dtype=np.int64)

def sample_error(n: int, bound: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Sample small bounded error polynomial from [-bound, bound] (e <- psi)."""
    if rng is None:
        rng = np.random.default_rng()
    return rng.integers(-bound, bound + 1, size=n, dtype=np.int64)

def sample_smudging_noise(n: int, bound: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Sample smudging noise polynomial from [-bound, bound] (e* <- phi)."""
    if rng is None:
        rng = np.random.default_rng()
    return rng.integers(-bound, bound + 1, size=n, dtype=np.int64)

def poly_zero(n: int) -> np.ndarray:
    """Create the zero polynomial in R_q."""
    return np.zeros(n, dtype=np.int64)

def poly_from_int(value: int, n: int) -> np.ndarray:
    """Embed an integer as a constant polynomial in R_q."""
    poly = np.zeros(n, dtype=np.int64)
    poly[0] = value
    return poly

def poly_norm(poly: np.ndarray) -> float:
    """Compute infinity norm (max absolute coefficient)."""
    return float(np.max(np.abs(poly)))

def mod_inverse(a: int, q: int) -> int:
    """Compute modular inverse a^(-1) mod q via Fermat's Little Theorem."""
    a = a % q
    if a == 0:
        raise ValueError(f"Cannot invert 0 modulo {q}")
    return pow(int(a), q - 2, q)

def poly_coeff_mod_inverse(coeff: int, q: int) -> int:
    """Modular inverse for constant ring elements."""
    return mod_inverse(coeff, q)

def encode_vector_as_polynomials(vec: np.ndarray, n: int, q: int, scale: int = 1) -> List[np.ndarray]:
    """Quantize float vector and pack into a list of degree-n polynomials."""
    quantized = np.round(vec * scale).astype(np.int64)
    remainder = len(quantized) % n
    if remainder != 0:
        quantized = np.concatenate([quantized, np.zeros(n - remainder, dtype=np.int64)])
    num_polys = len(quantized) // n
    polynomials = []
    for i in range(num_polys):
        poly = quantized[i * n : (i + 1) * n]
        polynomials.append(poly_mod_q(poly, q))
    return polynomials

def decode_polynomials_to_vector(polynomials: List[np.ndarray], original_length: int, q: int, scale: int = 1) -> np.ndarray:
    """Unpack polynomials back to float vector."""
    all_coeffs = np.concatenate([poly_mod_q(p, q) for p in polynomials])
    all_coeffs = all_coeffs[:original_length]
    return all_coeffs.astype(np.float64) / scale

print("Ring arithmetic module defined successfully.")


In [ ]:
# Quick Ring Verification: X^(n-1) * X = X^n == -1
test_n, test_q = 4, 1048583
x_cubed = np.array([0, 0, 0, 1], dtype=np.int64) # X^3
x_one = np.array([0, 1, 0, 0], dtype=np.int64)   # X
x_fourth = poly_mul(x_cubed, x_one, test_q)
print(f"Negacyclic test: X^3 * X mod (X^4+1) = {x_fourth} (Expected: [-1, 0, 0, 0])")
assert x_fourth[0] == -1 and np.all(x_fourth[1:] == 0), "Negacyclic reduction error!"


## 2. Ring Learning With Errors (RLWE)

### Mathematical Foundation (Section 2.2)
The security of lattice-based homomorphic encryption relies on the **Ring Learning With Errors (RLWE)** problem:
- $a \leftarrow R_q$ is a uniformly random public polynomial.
- $s \leftarrow \chi$ is a secret key with small ternary coefficients $\{-1, 0, 1\}$.
- $e \leftarrow \psi$ is a small error polynomial with bounded coefficients in $[-\text{bound}, \text{bound}]$.
- An RLWE sample is:
$$b = a \cdot s + e \pmod q$$

Under the RLWE hardness assumption, the pair $(a, b)$ is computationally indistinguishable from uniform randomness $(a, u) \in R_q^2$.


In [ ]:
# ============================================================================
# RING LEARNING WITH ERRORS (RLWE)
# ============================================================================

def sample_secret(n: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Sample secret key s <- chi with ternary coefficients."""
    return sample_ternary(n, rng)

def sample_error_poly(n: int, bound: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Sample error polynomial e <- psi with bounded coefficients."""
    return sample_error(n, bound, rng)

def sample_uniform_polynomial(n: int, q: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Sample uniform public polynomial a <- R_q."""
    return sample_uniform(n, q, rng)

def generate_rlwe_sample(s: np.ndarray, a: np.ndarray, q: int, error_bound: int,
                         rng: Optional[np.random.Generator] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Generate RLWE sample (a, b = a*s + e mod q)."""
    n = len(s)
    e = sample_error_poly(n, error_bound, rng)
    a_times_s = poly_mul(a, s, q)
    b = poly_add(a_times_s, e, q)
    return a, b

def verify_rlwe_structure(a: np.ndarray, b: np.ndarray, s: np.ndarray, q: int) -> np.ndarray:
    """Recover noise e = b - a*s mod q."""
    a_times_s = poly_mul(a, s, q)
    return poly_sub(b, a_times_s, q)

def demo_rlwe():
    """RLWE verification demonstration."""
    print("=" * 60)
    print("RLWE DEMONSTRATION")
    print("=" * 60)
    n, q, bound = 8, 1048583, 3
    rng = np.random.default_rng(42)
    s = sample_secret(n, rng)
    a = sample_uniform_polynomial(n, q, rng)
    _, b = generate_rlwe_sample(s, a, q, bound, rng)
    rec_e = verify_rlwe_structure(a, b, s, q)
    print(f"Secret s:        {s}")
    print(f"Sample b:        {b}")
    print(f"Recovered e:     {rec_e} (Max abs: {np.max(np.abs(rec_e))} <= {bound})")
    assert np.max(np.abs(rec_e)) <= bound, "Error exceeds bound!"

demo_rlwe()


## 3. CDKS Multi-Key Homomorphic Encryption

### Mathematical Construction (Section 2.4)
The CDKS (Chen-Dai-Kim-Song) scheme allows individual encryption and homomorphic addition across distinct keys:

1. **Setup:** $pp = (n, q, \chi, \psi, a)$ where $a \leftarrow R_q$ is globally shared.
2. **KeyGen:**
   $$s_i \leftarrow \chi, \quad e_i \leftarrow \psi$$
   $$b_i = -a \cdot s_i + e_i \pmod q$$
   $$sk_i = (1, s_i), \quad pk_i = (b_i, a)$$
3. **Enc:** For plaintext polynomial $m_i$:
   $$v_i \leftarrow \chi, \quad e_{i,0}, e_{i,1} \leftarrow \psi$$
   $$c_{i,0} = v_i \cdot b_i + m_i + e_{i,0} \pmod q$$
   $$c_{i,1} = v_i \cdot a + e_{i,1} \pmod q$$
4. **Add:**
   $$c_0 = \sum_{i=1}^N c_{i,0}, \quad ct_{\text{add}} = (c_0, c_{1,1}, c_{2,1}, \dots, c_{N,1})$$
5. **Partial Decrypt:** $\mu_i = c_{i,1} \cdot s_i + e_i^*$ where $e_i^* \leftarrow \phi$ (smudging noise).
6. **Merge:** $M = c_0 + \sum_{i=1}^N \mu_i \approx \sum_{i=1}^N m_i$.


In [ ]:
# ============================================================================
# CDKS MULTI-KEY HOMOMORPHIC ENCRYPTION
# ============================================================================

class CDKSPublicParams:
    def __init__(self, n: int, q: int, error_bound: int, smudge_bound: int, a: np.ndarray):
        self.n = n
        self.q = q
        self.error_bound = error_bound
        self.smudge_bound = smudge_bound
        self.a = a

def cdks_setup(n: int, q: int, error_bound: int = 3, smudge_bound: int = 50,
               rng: Optional[np.random.Generator] = None) -> CDKSPublicParams:
    if rng is None:
        rng = np.random.default_rng()
    a = sample_uniform_polynomial(n, q, rng)
    return CDKSPublicParams(n, q, error_bound, smudge_bound, a)

def cdks_keygen(pp: CDKSPublicParams, rng: Optional[np.random.Generator] = None) -> Tuple[dict, dict]:
    if rng is None:
        rng = np.random.default_rng()
    n, q, a = pp.n, pp.q, pp.a
    s_i = sample_secret(n, rng)
    e_i = sample_error_poly(n, pp.error_bound, rng)
    a_times_s = poly_mul(a, s_i, q)
    neg_a_times_s = poly_mod_q(-a_times_s, q)
    b_i = poly_add(neg_a_times_s, e_i, q)
    sk_i = {'one': 1, 's': s_i}
    pk_i = {'b': b_i, 'a': a.copy()}
    return sk_i, pk_i

def cdks_encrypt(pp: CDKSPublicParams, pk: dict, m: np.ndarray,
                 rng: Optional[np.random.Generator] = None) -> Tuple[np.ndarray, np.ndarray]:
    if rng is None:
        rng = np.random.default_rng()
    n, q = pp.n, pp.q
    b_i, a = pk['b'], pk['a']
    v_i = sample_secret(n, rng)
    e_i0 = sample_error_poly(n, pp.error_bound, rng)
    e_i1 = sample_error_poly(n, pp.error_bound, rng)
    v_times_b = poly_mul(v_i, b_i, q)
    c_i0 = poly_add(poly_add(v_times_b, m, q), e_i0, q)
    v_times_a = poly_mul(v_i, a, q)
    c_i1 = poly_add(v_times_a, e_i1, q)
    return c_i0, c_i1

def cdks_add(ciphertexts: List[Tuple[np.ndarray, np.ndarray]], q: int) -> Tuple[np.ndarray, List[np.ndarray]]:
    n = len(ciphertexts[0][0])
    c_0_sum = poly_zero(n)
    for c_i0, c_i1 in ciphertexts:
        c_0_sum = poly_add(c_0_sum, c_i0, q)
    c1_list = [c_i1 for _, c_i1 in ciphertexts]
    return c_0_sum, c1_list

def cdks_partial_decrypt(sk: dict, c_i1: np.ndarray, pp: CDKSPublicParams,
                         rng: Optional[np.random.Generator] = None) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    s_i = sk['s']
    c_times_s = poly_mul(c_i1, s_i, pp.q)
    e_star = sample_smudging_noise(pp.n, pp.smudge_bound, rng)
    return poly_add(c_times_s, e_star, pp.q)

def cdks_merge(c_0_sum: np.ndarray, partial_decryptions: List[np.ndarray], q: int) -> np.ndarray:
    n = len(c_0_sum)
    mu_sum = poly_zero(n)
    for mu_i in partial_decryptions:
        mu_sum = poly_add(mu_sum, mu_i, q)
    return poly_add(c_0_sum, mu_sum, q)

def full_cdks_pipeline(pp: CDKSPublicParams, plaintexts: List[np.ndarray],
                       rng: Optional[np.random.Generator] = None) -> dict:
    if rng is None:
        rng = np.random.default_rng()
    N, q = len(plaintexts), pp.q
    keys = [cdks_keygen(pp, rng) for _ in range(N)]
    ciphertexts = [cdks_encrypt(pp, keys[i][1], plaintexts[i], rng) for i in range(N)]
    c_0_sum, c1_list = cdks_add(ciphertexts, q)
    partial_decs = [cdks_partial_decrypt(keys[i][0], c1_list[i], pp, rng) for i in range(N)]
    M_recovered = cdks_merge(c_0_sum, partial_decs, q)
    M_true = poly_zero(pp.n)
    for m in plaintexts:
        M_true = poly_add(M_true, m, q)
    return {
        'keys': keys,
        'ciphertexts': ciphertexts,
        'c_0_sum': c_0_sum,
        'c1_list': c1_list,
        'partial_decryptions': partial_decs,
        'M_recovered': M_recovered,
        'M_true': M_true,
        'max_error': int(poly_norm(poly_mod_q(M_recovered - M_true, q)))
    }

def demo_cdks():
    print("=" * 60)
    print("CDKS MULTI-KEY HE DEMONSTRATION")
    print("=" * 60)
    n, q = 8, 1048583
    rng = np.random.default_rng(42)
    pp = cdks_setup(n, q, error_bound=3, smudge_bound=10, rng=rng)
    m1 = np.array([10, 20, 30, 0, 0, 0, 0, 0], dtype=np.int64)
    m2 = np.array([5, 15, 25, 0, 0, 0, 0, 0], dtype=np.int64)
    m3 = np.array([3, 7, 11, 0, 0, 0, 0, 0], dtype=np.int64)
    res = full_cdks_pipeline(pp, [m1, m2, m3], rng)
    print(f"Plaintexts sum expected: {m1[:3] + m2[:3] + m3[:3]}")
    print(f"Recovered aggregate M:   {res['M_recovered'][:3]}")
    print(f"Max noise error:         {res['max_error']}")
    assert res['max_error'] < 500, "Decryption noise unexpectedly high!"

demo_cdks()


## 4. Shamir Linear Secret Sharing (Classic & Over $R_q$) & Re-Sharing

### Mathematical Principles (Sections 2.5, 2.6, 2.7)
1. **Classic $(t, N)$-threshold sharing:**
   $$f(X) = s + \lambda_1 X + \lambda_2 X^2 + \dots + \lambda_{t-1} X^{t-1} \pmod p$$
   Reconstruction via Lagrange basis polynomials:
   $$s = f(0) = \sum_{i \in S} f(\alpha_i) \cdot \prod_{j \in S, j \ne i} \frac{-\alpha_j}{\alpha_i - \alpha_j} \pmod p$$

2. **Ring Shamir over $R_q$:**
   The secret $s$ and random coefficients $\lambda_k$ are polynomials in $R_q$.
   **Definition 2 (Exceptional Sequence):** Points $\alpha_1, \dots, \alpha_N$ must satisfy:
   $$\alpha_i - \alpha_j \in R_q^\times \quad \forall i \ne j$$
   When $\alpha_i$ are distinct integers, $\gcd(\alpha_i - \alpha_j, q) = 1$ ensures units in $R_q$.

3. **Share Re-sharing:**
   Participants collectively reconstruct $S = \sum_{i=1}^N s_i$ without exposing any individual $s_i$:
   - Client $i$ shares $s_i$ via $f_i(X)$.
   - Client $j$ receives $f_i(\alpha_j)$ from all $i$, computing $\tilde{s}_j = \sum_{i=1}^N f_i(\alpha_j)$.
   - Lagrange interpolation on $\{(\alpha_j, \tilde{s}_j)\}_{j=1}^t$ yields $\sum f_i(0) = \sum s_i$.


In [ ]:
# ============================================================================
# SHAMIR LINEAR SECRET SHARING (CLASSIC & OVER R_q)
# ============================================================================

class ClassicShamir:
    @staticmethod
    def share(secret: int, t: int, alphas: List[int], p: int,
              rng: Optional[np.random.Generator] = None) -> List[Tuple[int, int]]:
        if rng is None:
            rng = np.random.default_rng()
        coeffs = [secret % p] + [int(rng.integers(0, p)) for _ in range(t - 1)]
        shares = []
        for alpha in alphas:
            val = 0
            for j in range(len(coeffs) - 1, -1, -1):
                val = (val * alpha + coeffs[j]) % p
            shares.append((alpha, val))
        return shares

    @staticmethod
    def combine(shares: List[Tuple[int, int]], p: int) -> int:
        secret = 0
        k = len(shares)
        for i in range(k):
            alpha_i, y_i = shares[i]
            num, den = 1, 1
            for j in range(k):
                if j == i: continue
                alpha_j = shares[j][0]
                num = (num * (-alpha_j)) % p
                den = (den * (alpha_i - alpha_j)) % p
            lagrange = (num * mod_inverse(den, p)) % p
            secret = (secret + y_i * lagrange) % p
        return secret

class RingShamir:
    @staticmethod
    def check_exceptional_sequence(alphas: List[int], q: int) -> bool:
        from math import gcd
        N = len(alphas)
        for i in range(N):
            for j in range(i + 1, N):
                diff = (alphas[i] - alphas[j]) % q
                if diff == 0 or gcd(abs(int(diff)), int(q)) != 1:
                    return False
        return True

    @staticmethod
    def share(secret_poly: np.ndarray, t: int, alphas: List[int], n_ring: int, q: int,
              rng: Optional[np.random.Generator] = None) -> List[Tuple[int, np.ndarray]]:
        if rng is None:
            rng = np.random.default_rng()
        coeffs = [secret_poly.copy()] + [sample_uniform(n_ring, q, rng) for _ in range(t - 1)]
        shares = []
        for alpha in alphas:
            res = poly_zero(n_ring)
            for j in range(len(coeffs) - 1, -1, -1):
                res = poly_scalar_mul(res, alpha, q)
                res = poly_add(res, coeffs[j], q)
            shares.append((alpha, res))
        return shares

    @staticmethod
    def combine(shares: List[Tuple[int, np.ndarray]], n_ring: int, q: int) -> np.ndarray:
        k = len(shares)
        secret = poly_zero(n_ring)
        for i in range(k):
            alpha_i, y_i = shares[i]
            num, den = 1, 1
            for j in range(k):
                if j == i: continue
                alpha_j = shares[j][0]
                num = (num * (-alpha_j)) % q
                den = (den * (alpha_i - alpha_j)) % q
            lagrange = (num * mod_inverse(den, q)) % q
            term = poly_scalar_mul(y_i, lagrange, q)
            secret = poly_add(secret, term, q)
        return secret

def reshare(secrets: List[np.ndarray], t: int, alphas: List[int], n_ring: int, q: int,
            rng: Optional[np.random.Generator] = None) -> List[Tuple[int, np.ndarray]]:
    """Share re-sharing protocol: reconstruct sum of secrets without exposing individuals."""
    if rng is None:
        rng = np.random.default_rng()
    N = len(secrets)
    all_shares = [RingShamir.share(secrets[i], t, alphas, n_ring, q, rng) for i in range(N)]
    aggregated_shares = []
    for j in range(N):
        alpha_j = alphas[j]
        agg_share = poly_zero(n_ring)
        for i in range(N):
            _, val = all_shares[i][j]
            agg_share = poly_add(agg_share, val, q)
        aggregated_shares.append((alpha_j, agg_share))
    return aggregated_shares

def demo_shamir_all():
    print("=" * 60)
    print("SHAMIR SECRET SHARING & RE-SHARING DEMO")
    print("=" * 60)
    # 1. Classic Shamir
    p, t, N, secret = 104729, 3, 5, 42
    alphas = list(range(1, N + 1))
    c_shares = ClassicShamir.share(secret, t, alphas, p)
    rec_c = ClassicShamir.combine(c_shares[:t], p)
    print(f"Classic Shamir (t={t}, N={N}): Secret={secret}, Recovered={rec_c}")
    assert rec_c == secret

    # 2. Ring Shamir
    n_ring, q = 8, 1048583
    sec_poly = np.array([10, 20, 30, 40, 50, 60, 70, 80], dtype=np.int64)
    r_shares = RingShamir.share(sec_poly, t, alphas, n_ring, q)
    rec_r = RingShamir.combine(r_shares[:t], n_ring, q)
    assert np.array_equal(poly_mod_q(rec_r, q), poly_mod_q(sec_poly, q))
    print(f"Ring Shamir: Secret={sec_poly[:3]}..., Recovered={rec_r[:3]}...")

    # 3. Re-sharing
    s1 = np.array([10, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    s2 = np.array([20, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    s3 = np.array([30, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    agg_shares = reshare([s1, s2, s3], t=2, alphas=[1, 2, 3], n_ring=8, q=q)
    rec_sum = RingShamir.combine(agg_shares[:2], n_ring=8, q=q)
    print(f"Re-sharing: s1=10, s2=20, s3=30 -> Reconstructed Sum = {rec_sum[0]} (Expected: 60)")
    assert rec_sum[0] == 60

demo_shamir_all()


## 5. The Proposed CDKS-LSS Scheme

### Algorithm 2 Walkthrough (Section 4.2)
1. **Partial Decryption:** Each client $i$ calculates $\mu_i = c_{i,1} \cdot s_i \pmod q$.
2. **Polynomial Creation:** Client $i$ constructs a degree $t-1$ polynomial $f_i(X) \in R_q[X]$ with $f_i(0) = \mu_i$.
3. **P2P Share Distribution:** Client $i$ securely transmits $f_i(\alpha_j)$ to client $j$.
4. **Local Aggregation:** Client $j$ sums all incoming shares: $\tilde{s}_j = \sum_{i=1}^N f_i(\alpha_j)$.
5. **Reconstruction:** The server gathers any $t$ shares $\{(\alpha_j, \tilde{s}_j)\}$ and interpolates $F(0) = \sum_{i=1}^N \mu_i$.
6. **Plaintext Sum:** Server adds $M = c_0 + \sum \mu_i \approx \sum m_i$.


In [ ]:
# ============================================================================
# CDKS-LSS PROTOCOL (SECTION 4.2, ALGORITHM 2)
# ============================================================================

def dec_lss(secret_keys: List[dict], c_0_sum: np.ndarray, c1_list: List[np.ndarray],
            pp: CDKSPublicParams, t: int, alphas: List[int],
            available_indices: Optional[List[int]] = None,
            rng: Optional[np.random.Generator] = None) -> dict:
    if rng is None:
        rng = np.random.default_rng()
    N, n_ring, q = len(secret_keys), pp.n, pp.q
    if available_indices is None:
        available_indices = list(range(N))

    # Step 1: Compute mu_i = c_{i,1} * s_i
    mu_values = [poly_mul(c1_list[i], secret_keys[i]['s'], q) for i in range(N)]

    # Steps 2-3: Secret share mu_i
    all_shares = [RingShamir.share(mu_values[i], t, alphas, n_ring, q, rng) for i in range(N)]

    # Step 4: Each participant aggregates received shares: s_tilde_j = sum f_i(alpha_j)
    aggregated_shares = []
    for j in range(N):
        alpha_j = alphas[j]
        agg_share = poly_zero(n_ring)
        for i in range(N):
            _, share_val = all_shares[i][j]
            agg_share = poly_add(agg_share, share_val, q)
        aggregated_shares.append((alpha_j, agg_share))

    # Steps 5-6: Server collects available shares and reconstructs sum mu_i
    available_shares = [aggregated_shares[i] for i in available_indices]
    if len(available_shares) < t:
        raise ValueError(f"Insufficient shares! Have {len(available_shares)}, need {t}.")
    used_shares = available_shares[:t]
    mu_sum = RingShamir.combine(used_shares, n_ring, q)

    # Step 7: Final aggregation: M = c_0 + sum mu_i
    M_recovered = poly_add(c_0_sum, mu_sum, q)
    return {
        'M_recovered': M_recovered,
        'mu_values': mu_values,
        'agg_shares': aggregated_shares,
        'used_shares': used_shares,
        'mu_sum': mu_sum
    }

def full_cdks_lss_pipeline(pp: CDKSPublicParams, plaintexts: List[np.ndarray], t: int,
                           alphas: List[int], available_indices: Optional[List[int]] = None,
                           rng: Optional[np.random.Generator] = None) -> dict:
    if rng is None:
        rng = np.random.default_rng()
    N, q = len(plaintexts), pp.q
    if available_indices is None:
        available_indices = list(range(N))
    keys = [cdks_keygen(pp, rng) for _ in range(N)]
    ciphertexts = [cdks_encrypt(pp, keys[i][1], plaintexts[i], rng) for i in range(N)]
    c_0_sum, c1_list = cdks_add(ciphertexts, q)
    secret_keys = [keys[i][0] for i in range(N)]
    dec_res = dec_lss(secret_keys, c_0_sum, c1_list, pp, t, alphas, available_indices, rng)
    M_true = poly_zero(pp.n)
    for m in plaintexts:
        M_true = poly_add(M_true, m, q)
    err = poly_mod_q(dec_res['M_recovered'] - M_true, q)
    return {
        'keys': keys,
        'ciphertexts': ciphertexts,
        'c_0_sum': c_0_sum,
        'c1_list': c1_list,
        'M_recovered': dec_res['M_recovered'],
        'M_true': M_true,
        'decryption_error': err,
        'max_error': int(poly_norm(err)),
        'mu_values': dec_res['mu_values'],
        'agg_shares': dec_res['agg_shares'],
        'used_shares': dec_res['used_shares']
    }

def demo_cdks_lss():
    print("=" * 60)
    print("CDKS-LSS DEMONSTRATION (N=3, t=2)")
    print("=" * 60)
    n, q, t, N = 8, 1048583, 2, 3
    alphas = [1, 2, 3]
    rng = np.random.default_rng(42)
    pp = cdks_setup(n, q, error_bound=3, smudge_bound=10, rng=rng)
    m1 = np.array([100, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    m2 = np.array([200, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    m3 = np.array([300, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    res = full_cdks_lss_pipeline(pp, [m1, m2, m3], t, alphas, rng=rng)
    print(f"Server recovered sum: {res['M_recovered'][0]} (Expected ~600)")
    print(f"Max error:            {res['max_error']}")
    print(f"Individual mu_i:      mu_1={res['mu_values'][0][0]}, mu_2={res['mu_values'][1][0]}, mu_3={res['mu_values'][2][0]} (Hidden from server!)")
    assert res['max_error'] < 500

demo_cdks_lss()


## 6. Baseline xMK-CKKS Scheme

### Overview & Limitations (Section 3.2)
In the xMK-CKKS baseline:
- All participants encrypt under an **aggregated public key**: $\tilde{b} = \sum_{i=1}^N b_i$.
- Ciphertexts are aggregated directly: $(c_0, c_1) = (\sum c_{i,0}, \sum c_{i,1})$.
- **Fatal flaw:** Decryption requires partial decryption contributions from **all $N$ participants**. If even a single client drops out, the combined secret $\sum s_i$ cannot be matched and decryption fails completely.


In [ ]:
# ============================================================================
# BASELINE xMK-CKKS (SECTION 3.2)
# ============================================================================

def xmk_generate_keys(pp: CDKSPublicParams, N: int, rng: Optional[np.random.Generator] = None) -> dict:
    if rng is None:
        rng = np.random.default_rng()
    individual_keys = [cdks_keygen(pp, rng) for _ in range(N)]
    b_tilde = poly_zero(pp.n)
    for sk_i, pk_i in individual_keys:
        b_tilde = poly_add(b_tilde, pk_i['b'], pp.q)
    pk_agg = {'b': b_tilde, 'a': pp.a.copy()}
    return {'individual_keys': individual_keys, 'pk_agg': pk_agg, 'b_tilde': b_tilde}

def xmk_encrypt(pp: CDKSPublicParams, pk_agg: dict, m: np.ndarray,
                rng: Optional[np.random.Generator] = None) -> Tuple[np.ndarray, np.ndarray]:
    if rng is None:
        rng = np.random.default_rng()
    v = sample_secret(pp.n, rng)
    e_0 = sample_error_poly(pp.n, pp.error_bound, rng)
    e_1 = sample_error_poly(pp.n, pp.error_bound, rng)
    c_0 = poly_add(poly_add(poly_mul(v, pk_agg['b'], pp.q), m, pp.q), e_0, pp.q)
    c_1 = poly_add(poly_mul(v, pk_agg['a'], pp.q), e_1, pp.q)
    return c_0, c_1

def xmk_aggregate(ciphertexts: List[Tuple[np.ndarray, np.ndarray]], q: int) -> Tuple[np.ndarray, np.ndarray]:
    n = len(ciphertexts[0][0])
    c0_sum, c1_sum = poly_zero(n), poly_zero(n)
    for c0, c1 in ciphertexts:
        c0_sum = poly_add(c0_sum, c0, q)
        c1_sum = poly_add(c1_sum, c1, q)
    return c0_sum, c1_sum

def xmk_partial_decrypt(sk: dict, c_1: np.ndarray, pp: CDKSPublicParams,
                        rng: Optional[np.random.Generator] = None) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    c_times_s = poly_mul(c_1, sk['s'], pp.q)
    e_star = sample_smudging_noise(pp.n, pp.smudge_bound, rng)
    return poly_add(c_times_s, e_star, pp.q)

def xmk_merge(c_0_sum: np.ndarray, partial_decryptions: List[np.ndarray], q: int) -> np.ndarray:
    n = len(c_0_sum)
    mu_sum = poly_zero(n)
    for mu_i in partial_decryptions:
        mu_sum = poly_add(mu_sum, mu_i, q)
    return poly_add(c_0_sum, mu_sum, q)

def full_xmk_ckks_pipeline(pp: CDKSPublicParams, plaintexts: List[np.ndarray],
                           available_indices: Optional[List[int]] = None,
                           rng: Optional[np.random.Generator] = None) -> dict:
    if rng is None:
        rng = np.random.default_rng()
    N, q = len(plaintexts), pp.q
    if available_indices is None:
        available_indices = list(range(N))
    key_data = xmk_generate_keys(pp, N, rng)
    ciphertexts = [xmk_encrypt(pp, key_data['pk_agg'], m, rng) for m in plaintexts]
    c0_sum, c1_sum = xmk_aggregate(ciphertexts, q)
    partial_decs = [xmk_partial_decrypt(key_data['individual_keys'][i][0], c1_sum, pp, rng) for i in available_indices]
    M_rec = xmk_merge(c0_sum, partial_decs, q)
    M_true = poly_zero(pp.n)
    for m in plaintexts:
        M_true = poly_add(M_true, m, q)
    err = poly_mod_q(M_rec - M_true, q)
    return {
        'M_recovered': M_rec,
        'M_true': M_true,
        'max_error': int(poly_norm(err)),
        'success': len(available_indices) == N
    }

def demo_xmk_ckks():
    print("=" * 60)
    print("xMK-CKKS BASELINE DEMONSTRATION")
    print("=" * 60)
    n, q, N = 8, 1048583, 3
    rng = np.random.default_rng(42)
    pp = cdks_setup(n, q, error_bound=3, smudge_bound=10, rng=rng)
    m = [np.array([100*i, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64) for i in [1, 2, 3]]
    res_all = full_xmk_ckks_pipeline(pp, m, rng=rng)
    print(f"All {N} clients present -> Success: {res_all['max_error'] < 1000} (Error: {res_all['max_error']})")
    
    rng2 = np.random.default_rng(42)
    pp2 = cdks_setup(n, q, error_bound=3, smudge_bound=10, rng=rng2)
    res_drop = full_xmk_ckks_pipeline(pp2, m, available_indices=[0, 1], rng=rng2)
    print(f"1 client dropped (2/{N})  -> Success: {res_drop['max_error'] < 1000} (Error: {res_drop['max_error']}) [FAILED as expected]")

demo_xmk_ckks()


## 7. Security Vulnerability & Attack Demonstrations

### Mathematical Vulnerability Derivations
1. **Plain FL:** Server observes client weights $w_i^r$ in the clear.
2. **CDKS Leakage Attack:**
   The server knows $c_{i,0} = v_i b_i + m_i + e_{i,0}$ and receives $\mu_i = c_{i,1} s_i + e_i^*$.
   Because $b_i = -a s_i + e_i$ and $c_{i,1} = v_i a + e_{i,1}$:
   $$c_{i,0} + \mu_i = v_i (-a s_i + e_i) + m_i + e_{i,0} + (v_i a + e_{i,1}) s_i + e_i^*$$
   $$-v_i a s_i \text{ and } +v_i a s_i \text{ cancel out!}$$
   $$c_{i,0} + \mu_i = m_i + (v_i e_i + e_{i,0} + e_{i,1} s_i + e_i^*) \approx m_i$$
3. **CDKS-LSS Protection:**
   The server receives only $\tilde{s}_j = \sum_{i=1}^N f_i(\alpha_j)$, reconstructing $\sum \mu_i$. It never receives $\mu_i$, preventing the cancellation attack!


In [ ]:
# ============================================================================
# ATTACK DEMONSTRATIONS
# ============================================================================

def attack_plain_fl(model_updates: List[np.ndarray]) -> dict:
    print("\n[Attack 1: Plain FL] Server inspects raw updates:")
    for i, u in enumerate(model_updates):
        print(f"  Client {i+1} plaintext: {u[:4]}...")
    return {'attack': 'plain_fl', 'privacy': False}

def attack_cdks_partial_decryption(c_i0: np.ndarray, mu_i: np.ndarray,
                                   original_plaintext: np.ndarray, q: int) -> dict:
    recovered = poly_mod_q(c_i0 + mu_i, q)
    error = poly_mod_q(recovered - original_plaintext, q)
    max_err = int(poly_norm(error))
    success = max_err < 1000
    return {'recovered': recovered, 'max_error': max_err, 'success': success}

def run_all_attacks():
    print("=" * 60)
    print("COMPREHENSIVE THREE-ATTACK SECURITY EVALUATION")
    print("=" * 60)
    n, q = 8, 1048583
    m1 = np.array([100, 200, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    m2 = np.array([300, 400, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    m3 = np.array([500, 600, 0, 0, 0, 0, 0, 0], dtype=np.int64)
    plaintexts = [m1, m2, m3]

    # Attack 1: Plain FL
    attack_plain_fl(plaintexts)

    # Attack 2: CDKS
    print("\n[Attack 2: CDKS Leakage]")
    rng = np.random.default_rng(42)
    pp = cdks_setup(n, q, error_bound=3, smudge_bound=10, rng=rng)
    cdks_res = full_cdks_pipeline(pp, plaintexts, rng)
    for i in range(3):
        res = attack_cdks_partial_decryption(cdks_res['ciphertexts'][i][0], cdks_res['partial_decryptions'][i], plaintexts[i], q)
        print(f"  Client {i+1}: Recovered={res['recovered'][:2]}, Target={plaintexts[i][:2]}, Error={res['max_error']} -> EXPOSED!")

    # Attack 3: CDKS-LSS
    print("\n[Attack 3: CDKS-LSS Defense]")
    rng3 = np.random.default_rng(42)
    pp3 = cdks_setup(n, q, error_bound=3, smudge_bound=10, rng=rng3)
    lss_res = full_cdks_lss_pipeline(pp3, plaintexts, t=2, alphas=[1, 2, 3], rng=rng3)
    print(f"  Server recovers aggregate: {lss_res['M_recovered'][:2]} (Expected: {m1[:2]+m2[:2]+m3[:2]})")
    # If server attempts attack using aggregate sum
    fake_rec = poly_mod_q(lss_res['ciphertexts'][0][0] + lss_res['M_recovered'], q)
    print(f"  Attempted individual recovery error: {poly_norm(poly_mod_q(fake_rec - m1, q))} >> 1000 -> PROTECTED!")

run_all_attacks()


## 8. Machine Learning Model (Logistic Regression)

A pure NumPy logistic regression model for binary classification:
$$P(y=1|x) = \sigma(x \cdot w), \quad \sigma(z) = \frac{1}{1 + e^{-z}}$$
$$L(w) = -\frac{1}{M} \sum_{j=1}^M \left[ y_j \log(\hat{y}_j) + (1-y_j) \log(1-\hat{y}_j) \right]$$
$$\nabla L(w) = \frac{1}{M} X^T (\sigma(Xw) - y)$$


In [ ]:
# ============================================================================
# MACHINE LEARNING MODEL (LOGISTIC REGRESSION)
# ============================================================================

def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def predict_proba(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    return sigmoid(X @ w)

def predict(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (predict_proba(X, w) >= 0.5).astype(int)

def compute_loss(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    eps = 1e-15
    proba = np.clip(predict_proba(X, w), eps, 1 - eps)
    return float(-np.mean(y * np.log(proba) + (1 - y) * np.log(1 - proba)))

def compute_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    n_samples = len(y)
    return (1.0 / n_samples) * (X.T @ (predict_proba(X, w) - y))

def train(X: np.ndarray, y: np.ndarray, w: np.ndarray, learning_rate: float = 0.1, epochs: int = 5) -> np.ndarray:
    w = w.copy()
    for _ in range(epochs):
        grad = compute_gradient(X, y, w)
        w = w - learning_rate * grad
    return w

def initialize_weights(n_features: int, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    return rng.normal(0, 1.0 / np.sqrt(n_features), size=n_features)

print("Logistic regression model defined.")


## 9. Synthetic Dataset Generation & Non-IID Partitioning

Simulates real-world FL scenarios with heterogeneous non-IID client partitions using the pathological shard method (McMahan et al., 2017).


In [ ]:
# ============================================================================
# DATA GENERATION & NON-IID SPLITTING
# ============================================================================

def generate_synthetic_data(n_samples: int = 500, n_features: int = 20, n_classes: int = 2,
                            random_state: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    X, y = make_classification(
        n_samples=n_samples, n_features=n_features, n_informative=n_features // 2,
        n_redundant=n_features // 4, n_classes=n_classes, class_sep=1.0,
        random_state=random_state, flip_y=0.05
    )
    return X.astype(np.float64), y.astype(np.int64)

def split_iid(X: np.ndarray, y: np.ndarray, n_clients: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    indices = np.random.permutation(len(X))
    split_size = len(X) // n_clients
    return [(X[indices[i*split_size:(i+1)*split_size if i < n_clients-1 else len(X)]],
             y[indices[i*split_size:(i+1)*split_size if i < n_clients-1 else len(X)]])
            for i in range(n_clients)]

def split_non_iid(X: np.ndarray, y: np.ndarray, n_clients: int, shards_per_client: int = 2) -> List[Tuple[np.ndarray, np.ndarray]]:
    total_shards = n_clients * shards_per_client
    shard_size = len(X) // total_shards
    sorted_indices = np.argsort(y)
    shards = [sorted_indices[i * shard_size:(i + 1) * shard_size] for i in range(total_shards)]
    shard_indices = list(range(total_shards))
    np.random.shuffle(shard_indices)
    client_data = []
    for i in range(n_clients):
        c_shards = shard_indices[i * shards_per_client : (i + 1) * shards_per_client]
        idx = np.concatenate([shards[s] for s in c_shards])
        client_data.append((X[idx], y[idx]))
    return client_data

def generate_fl_dataset(n_clients: int = 5, n_samples_per_client: int = 100, n_features: int = 20,
                        non_iid: bool = True, random_state: int = 42):
    np.random.seed(random_state)
    total_train = n_clients * n_samples_per_client
    total_test = max(200, total_train // 5)
    X, y = generate_synthetic_data(total_train + total_test, n_features, random_state=random_state)
    X_train, X_test = X[:total_train], X[total_train:]
    y_train, y_test = y[:total_train], y[total_train:]
    client_data = split_non_iid(X_train, y_train, n_clients) if non_iid else split_iid(X_train, y_train, n_clients)
    return client_data, (X_test, y_test)

# Test generation
client_data_sample, test_data_sample = generate_fl_dataset(NUM_CLIENTS, NUM_SAMPLES_PER_CLIENT, NUM_FEATURES, non_iid=True)
print(f"Generated {NUM_CLIENTS} non-IID client partitions. Test set size: {len(test_data_sample[0])}")


## 10. Utility Functions & Communication Overhead Metrics

Provides weight-to-polynomial quantization and theoretical communication overhead modeling (Section 5.3).


In [ ]:
# ============================================================================
# UTILITIES & METRICS
# ============================================================================

def weights_to_polynomials(weights: np.ndarray, n: int, q: int, scale: int) -> List[np.ndarray]:
    return encode_vector_as_polynomials(weights, n, q, scale)

def polynomials_to_weights(polys: List[np.ndarray], original_length: int, q: int, scale: int) -> np.ndarray:
    return decode_polynomials_to_vector(polys, original_length, q, scale)

def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(y_true == y_pred))

class Timer:
    def __init__(self, name: str = ""):
        self.name = name
        self.elapsed = 0.0
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, *args):
        self.elapsed = time.time() - self.start

def communication_overhead(n_ring: int, q: int, N: int, t: int, num_weight_polys: int) -> Dict[str, dict]:
    coeff_bits = math.ceil(math.log2(q))
    coeff_bytes = math.ceil(coeff_bits / 8)
    poly_size = n_ring * coeff_bytes
    plaintext_model_size = num_weight_polys * n_ring * 8
    ciphertext_pair_size = 2 * num_weight_polys * poly_size
    ct_c0_size = num_weight_polys * poly_size
    share_size = num_weight_polys * poly_size
    pk_size = num_weight_polys * poly_size

    results = {}
    # FedAvg
    fedavg_c2s = N * plaintext_model_size
    fedavg_s2c = N * plaintext_model_size
    results['FedAvg'] = {
        'client_to_server_bytes': fedavg_c2s, 'client_to_client_bytes': 0,
        'server_to_client_bytes': fedavg_s2c, 'total_bytes': fedavg_c2s + fedavg_s2c,
        'client_to_server_msgs': N, 'client_to_client_msgs': 0, 'server_to_client_msgs': N,
        'total_msgs': 2 * N, 'ciphertext_size': 0, 'dropout_tolerance': 'Any (no crypto)',
        'individual_privacy': False
    }
    # xMK-CKKS
    xmk_c2s = N * ciphertext_pair_size + N * share_size
    xmk_s2c = N * plaintext_model_size + pk_size
    results['xMK-CKKS'] = {
        'client_to_server_bytes': xmk_c2s, 'client_to_client_bytes': 0,
        'server_to_client_bytes': xmk_s2c, 'total_bytes': xmk_c2s + xmk_s2c,
        'client_to_server_msgs': 2 * N, 'client_to_client_msgs': 0, 'server_to_client_msgs': N + 1,
        'total_msgs': 3 * N + 1, 'ciphertext_size': ciphertext_pair_size,
        'dropout_tolerance': 'None (all N required)', 'individual_privacy': False
    }
    # CDKS-LSS
    cdks_c2s = N * ct_c0_size + N * share_size
    cdks_c2c = N * (N - 1) * share_size
    cdks_s2c = N * plaintext_model_size
    results['CDKS-LSS'] = {
        'client_to_server_bytes': cdks_c2s, 'client_to_client_bytes': cdks_c2c,
        'server_to_client_bytes': cdks_s2c, 'total_bytes': cdks_c2s + cdks_c2c + cdks_s2c,
        'client_to_server_msgs': 2 * N, 'client_to_client_msgs': N * (N - 1),
        'server_to_client_msgs': N, 'total_msgs': 2 * N + N * (N - 1) + N,
        'ciphertext_size': ct_c0_size, 'dropout_tolerance': f'Up to {N - t} dropouts (need t={t})',
        'individual_privacy': True
    }
    return results

def format_overhead_table(results: Dict[str, dict]) -> str:
    def fmt_b(b):
        return f"{b / 1024:.1f} KB" if b < 1024*1024 else f"{b / (1024*1024):.1f} MB"
    lines = [f"\n{'Method':<12} | {'Client->Server':>14} | {'Client<->Client':>14} | {'Total Comm':>12} | {'Messages':>8} | {'Dropout':>22} | {'Privacy':>8}", "-" * 110]
    for m, d in results.items():
        lines.append(f"{m:<12} | {fmt_b(d['client_to_server_bytes']):>14} | {fmt_b(d['client_to_client_bytes']):>14} | {fmt_b(d['total_bytes']):>12} | {d['total_msgs']:>8} | {d['dropout_tolerance']:>22} | {'YES' if d['individual_privacy'] else 'NO':>8}")
    return "\n".join(lines)

print("Utilities and metrics defined.")


## 11. Federated Learning Entities (Client & Server)

Defines `FLClient` (local training, encryption, and share generation) and `FLServer` (homomorphic addition and LSS reconstruction).


In [ ]:
# ============================================================================
# FEDERATED LEARNING ENTITIES
# ============================================================================

class FLClient:
    def __init__(self, client_id: int, X: np.ndarray, y: np.ndarray):
        self.client_id = client_id
        self.X = X
        self.y = y
        self.sk = None
        self.pk = None

    def setup_crypto(self, pp: CDKSPublicParams, rng: Optional[np.random.Generator] = None):
        self.sk, self.pk = cdks_keygen(pp, rng)

    def local_train(self, global_weights: np.ndarray, learning_rate: float, local_epochs: int) -> np.ndarray:
        return train(self.X, self.y, global_weights, learning_rate, local_epochs)

    def encrypt_weights(self, weights: np.ndarray, pp: CDKSPublicParams, scale: int,
                        rng: Optional[np.random.Generator] = None) -> list:
        polys = weights_to_polynomials(weights, pp.n, pp.q, scale)
        return [cdks_encrypt(pp, self.pk, p, rng) for p in polys]

    def evaluate(self, weights: np.ndarray) -> Tuple[float, float]:
        y_pred = predict(self.X, weights)
        return float(np.mean(y_pred == self.y)), compute_loss(self.X, self.y, weights)

class FLServer:
    def __init__(self, initial_weights: np.ndarray):
        self.global_weights = initial_weights.copy()
        self.round = 0

    def fedavg_aggregate(self, client_weights: List[np.ndarray]) -> np.ndarray:
        self.global_weights = np.mean(client_weights, axis=0)
        self.round += 1
        return self.global_weights

    def cdks_aggregate_ciphertexts(self, all_ciphertexts: list, q: int) -> Tuple[list, list]:
        N, num_polys = len(all_ciphertexts), len(all_ciphertexts[0])
        c0_sums, c1_lists = [], []
        for k in range(num_polys):
            chunk_cts = [(all_ciphertexts[i][k][0], all_ciphertexts[i][k][1]) for i in range(N)]
            c0_sum, c1_list = cdks_add(chunk_cts, q)
            c0_sums.append(c0_sum)
            c1_lists.append(c1_list)
        return c0_sums, c1_lists

    def cdks_decrypt_aggregate(self, c0_sums: list, partial_decs_per_poly: list, q: int) -> list:
        return [cdks_merge(c0_sums[k], partial_decs_per_poly[k], q) for k in range(len(c0_sums))]

    def cdks_lss_reconstruct(self, c0_sums: list, agg_shares_per_poly: list, t: int,
                             n_ring: int, q: int, available_indices: Optional[List[int]] = None) -> list:
        recovered_polys = []
        for k in range(len(c0_sums)):
            shares_k = agg_shares_per_poly[k]
            if len(shares_k) < t:
                raise ValueError(f"Need {t} shares, have {len(shares_k)}")
            mu_sum_k = RingShamir.combine(shares_k[:t], n_ring, q)
            recovered_polys.append(poly_add(c0_sums[k], mu_sum_k, q))
        return recovered_polys

    def update_global_model_from_encrypted(self, recovered_polys: list, original_length: int,
                                           q: int, scale: int, N: int):
        weight_sum = polynomials_to_weights(recovered_polys, original_length, q, scale)
        self.global_weights = weight_sum / N
        self.round += 1
        return self.global_weights

print("FL Client and Server classes defined.")


## 12. Federated Learning Orchestration Loops

Four complete federated learning training procedures:
1. `run_fedavg` — Plain baseline (no crypto).
2. `run_cdks_fl` — CDKS encrypted FL (susceptible to partial decryption attack).
3. `run_xmk_ckks_fl` — xMK-CKKS baseline (no dropout tolerance).
4. `run_cdks_lss_fl` — Proposed CDKS-LSS framework (secure & dropout tolerant).


In [ ]:
# ============================================================================
# FEDERATED LEARNING PROTOCOL LOOPS
# ============================================================================

def run_fedavg(clients: List[FLClient], test_data: Tuple[np.ndarray, np.ndarray],
               n_features: int, n_rounds: int = 20, learning_rate: float = 0.1,
               local_epochs: int = 5, rng: Optional[np.random.Generator] = None) -> Dict:
    if rng is None: rng = np.random.default_rng()
    X_test, y_test = test_data
    w_global = initialize_weights(n_features, rng)
    server = FLServer(w_global)
    history = {'accuracy': [], 'loss': [], 'time': []}
    for r in range(n_rounds):
        t0 = time.time()
        client_weights = [c.local_train(server.global_weights, learning_rate, local_epochs) for c in clients]
        server.fedavg_aggregate(client_weights)
        elapsed = time.time() - t0
        acc = accuracy(y_test, predict(X_test, server.global_weights))
        loss = compute_loss(X_test, y_test, server.global_weights)
        history['accuracy'].append(acc); history['loss'].append(loss); history['time'].append(elapsed)
    return history

def run_cdks_fl(clients: List[FLClient], test_data: Tuple[np.ndarray, np.ndarray],
                pp: CDKSPublicParams, n_features: int, n_rounds: int = 20,
                learning_rate: float = 0.1, local_epochs: int = 5,
                weight_scale: int = 1000, rng: Optional[np.random.Generator] = None) -> Dict:
    if rng is None: rng = np.random.default_rng()
    X_test, y_test = test_data
    N = len(clients)
    for c in clients: c.setup_crypto(pp, rng)
    w_global = initialize_weights(n_features, rng)
    server = FLServer(w_global)
    history = {'accuracy': [], 'loss': [], 'time': []}
    for r in range(n_rounds):
        t0 = time.time()
        client_weights = [c.local_train(server.global_weights, learning_rate, local_epochs) for c in clients]
        all_cts = [clients[i].encrypt_weights(client_weights[i], pp, weight_scale, rng) for i in range(N)]
        c0_sums, c1_lists = server.cdks_aggregate_ciphertexts(all_cts, pp.q)
        num_polys = len(c0_sums)
        partial_decs = [[] for _ in range(num_polys)]
        for i, client in enumerate(clients):
            for k in range(num_polys):
                mu_ik = poly_add(poly_mul(c1_lists[k][i], client.sk['s'], pp.q),
                                 sample_smudging_noise(pp.n, pp.smudge_bound, rng), pp.q)
                partial_decs[k].append(mu_ik)
        rec_polys = server.cdks_decrypt_aggregate(c0_sums, partial_decs, pp.q)
        server.update_global_model_from_encrypted(rec_polys, n_features, pp.q, weight_scale, N)
        elapsed = time.time() - t0
        acc = accuracy(y_test, predict(X_test, server.global_weights))
        loss = compute_loss(X_test, y_test, server.global_weights)
        history['accuracy'].append(acc); history['loss'].append(loss); history['time'].append(elapsed)
    return history

def run_xmk_ckks_fl(clients: List[FLClient], test_data: Tuple[np.ndarray, np.ndarray],
                     pp: CDKSPublicParams, n_features: int, n_rounds: int = 20,
                     learning_rate: float = 0.1, local_epochs: int = 5,
                     weight_scale: int = 1000, rng: Optional[np.random.Generator] = None) -> Dict:
    if rng is None: rng = np.random.default_rng()
    X_test, y_test = test_data
    N = len(clients)
    key_data = xmk_generate_keys(pp, N, rng)
    server = FLServer(initialize_weights(n_features, rng))
    history = {'accuracy': [], 'loss': [], 'time': []}
    for r in range(n_rounds):
        t0 = time.time()
        client_weights = [c.local_train(server.global_weights, learning_rate, local_epochs) for c in clients]
        all_cts = []
        for i in range(N):
            polys = weights_to_polynomials(client_weights[i], pp.n, pp.q, weight_scale)
            all_cts.append([xmk_encrypt(pp, key_data['pk_agg'], p, rng) for p in polys])
        num_polys = len(all_cts[0])
        c0_sums, c1_sums = [], []
        for k in range(num_polys):
            chunk_cts = [(all_cts[i][k][0], all_cts[i][k][1]) for i in range(N)]
            c0, c1 = xmk_aggregate(chunk_cts, pp.q)
            c0_sums.append(c0); c1_sums.append(c1)
        rec_polys = []
        for k in range(num_polys):
            pdecs = [xmk_partial_decrypt(key_data['individual_keys'][i][0], c1_sums[k], pp, rng) for i in range(N)]
            rec_polys.append(xmk_merge(c0_sums[k], pdecs, pp.q))
        server.update_global_model_from_encrypted(rec_polys, n_features, pp.q, weight_scale, N)
        elapsed = time.time() - t0
        acc = accuracy(y_test, predict(X_test, server.global_weights))
        loss = compute_loss(X_test, y_test, server.global_weights)
        history['accuracy'].append(acc); history['loss'].append(loss); history['time'].append(elapsed)
    return history

def run_cdks_lss_fl(clients: List[FLClient], test_data: Tuple[np.ndarray, np.ndarray],
                    pp: CDKSPublicParams, n_features: int, t: int, alphas: List[int],
                    n_rounds: int = 20, learning_rate: float = 0.1, local_epochs: int = 5,
                    weight_scale: int = 1000, available_indices: Optional[List[int]] = None,
                    rng: Optional[np.random.Generator] = None) -> Dict:
    if rng is None: rng = np.random.default_rng()
    X_test, y_test = test_data
    N = len(clients)
    if available_indices is None: available_indices = list(range(N))
    for c in clients: c.setup_crypto(pp, rng)
    server = FLServer(initialize_weights(n_features, rng))
    history = {'accuracy': [], 'loss': [], 'time': []}
    for r in range(n_rounds):
        t0 = time.time()
        client_weights = [c.local_train(server.global_weights, learning_rate, local_epochs) for c in clients]
        all_cts = [clients[i].encrypt_weights(client_weights[i], pp, weight_scale, rng) for i in range(N)]
        c0_sums, c1_lists = server.cdks_aggregate_ciphertexts(all_cts, pp.q)
        num_polys = len(c0_sums)
        agg_shares_per_poly = []
        for k in range(num_polys):
            mu_values_k = [poly_mul(c1_lists[k][i], clients[i].sk['s'], pp.q) for i in range(N)]
            all_shares_k = [RingShamir.share(mu_values_k[i], t, alphas, pp.n, pp.q, rng) for i in range(N)]
            agg_shares_k = []
            for j in available_indices:
                alpha_j = alphas[j]
                agg_val = poly_zero(pp.n)
                for i in range(N):
                    agg_val = poly_add(agg_val, all_shares_k[i][j][1], pp.q)
                agg_shares_k.append((alpha_j, agg_val))
            agg_shares_per_poly.append(agg_shares_k)
        rec_polys = server.cdks_lss_reconstruct(c0_sums, agg_shares_per_poly, t, pp.n, pp.q, available_indices)
        server.update_global_model_from_encrypted(rec_polys, n_features, pp.q, weight_scale, N)
        elapsed = time.time() - t0
        acc = accuracy(y_test, predict(X_test, server.global_weights))
        loss = compute_loss(X_test, y_test, server.global_weights)
        history['accuracy'].append(acc); history['loss'].append(loss); history['time'].append(elapsed)
    return history

print("All FL orchestration routines compiled.")


## 13. Comprehensive Unit Test Suite

Runs all 23 unit tests verifying:
- Polynomial addition, subtraction, negacyclic multiplication, modular inverse, centered reduction, and float quantization round-trips.
- RLWE sampling structure and ternary distributions.
- CDKS key generation, encryption, homomorphic addition, decryption, and vulnerability demonstration.
- Shamir classic and ring secret sharing, exceptional sequence checks, and share re-sharing.
- CDKS-LSS threshold decryption and end-to-end mathematical validation ($N=3, t=2$).


In [ ]:
# ============================================================================
# COMPREHENSIVE UNIT TEST SUITE (23 TESTS)
# ============================================================================

def run_all_unit_tests():
    passed_tests = 0
    failed_tests = 0
    errors = []

    def check(name, fn):
        nonlocal passed_tests, failed_tests
        try:
            fn()
            print(f"  [PASS] {name}")
            passed_tests += 1
        except Exception as ex:
            print(f"  [FAIL] {name}: {ex}")
            failed_tests += 1
            errors.append((name, str(ex)))

    print("=" * 60)
    print("RUNNING UNIT TESTS")
    print("=" * 60)

    # 1-7: Ring Tests
    check("ring_addition", lambda: assert_true(np.array_equal(poly_add(np.array([1,2]), np.array([3,4]), 101), np.array([4,6]))))
    check("ring_subtraction", lambda: assert_true(np.array_equal(poly_sub(np.array([10,20]), np.array([1,2]), 101), np.array([9,18]))))
    check("ring_multiplication", lambda: assert_true(np.array_equal(poly_mul(np.array([1,1,0,0]), np.array([1,1,0,0]), 101), np.array([1,2,1,0]))))
    check("ring_negacyclic", lambda: assert_true(poly_mul(np.array([0,0,0,1]), np.array([0,1,0,0]), 101)[0] == -1))
    check("mod_q_centering", lambda: assert_true(np.array_equal(poly_mod_q(np.array([100, 50, 0, 51]), 101), np.array([-1, 50, 0, -50]))))
    check("mod_inverse", lambda: assert_true((42 * mod_inverse(42, 104729)) % 104729 == 1))
    check("encoding_roundtrip", lambda: assert_true(np.max(np.abs(np.array([0.5, -1.3, 2.7]) - decode_polynomials_to_vector(encode_vector_as_polynomials(np.array([0.5, -1.3, 2.7]), 8, 1048583, 1000), 3, 1048583, 1000))) < 0.01))

    # 8-9: RLWE Tests
    def test_rlwe():
        rng = np.random.default_rng(42)
        s = sample_secret(8, rng)
        a = sample_uniform_polynomial(8, 1048583, rng)
        _, b = generate_rlwe_sample(s, a, 1048583, 3, rng)
        assert poly_norm(verify_rlwe_structure(a, b, s, 1048583)) <= 3
    check("rlwe_generation", test_rlwe)
    check("secret_distribution", lambda: assert_true(set(sample_secret(64, np.random.default_rng(42)).tolist()).issubset({-1, 0, 1})))

    # 10-14: CDKS Tests
    def test_cdks_keys():
        pp = cdks_setup(8, 1048583, rng=np.random.default_rng(42))
        sk, pk = cdks_keygen(pp, np.random.default_rng(42))
        assert 's' in sk and 'b' in pk and len(sk['s']) == 8
    check("cdks_keygen", test_cdks_keys)

    def test_cdks_enc():
        pp = cdks_setup(8, 1048583, rng=np.random.default_rng(42))
        sk, pk = cdks_keygen(pp, np.random.default_rng(42))
        c0, c1 = cdks_encrypt(pp, pk, np.array([10, 0, 0, 0, 0, 0, 0, 0]), np.random.default_rng(42))
        assert len(c0) == 8 and len(c1) == 8
    check("cdks_encryption", test_cdks_enc)

    check("cdks_addition", lambda: assert_true(full_cdks_pipeline(cdks_setup(8, 1048583, 3, 10, np.random.default_rng(42)), [np.array([10,0,0,0,0,0,0,0]), np.array([20,0,0,0,0,0,0,0])], np.random.default_rng(42))['max_error'] < 500))
    check("cdks_decryption", lambda: assert_true(abs(full_cdks_pipeline(cdks_setup(8, 1048583, 3, 10, np.random.default_rng(42)), [np.array([100*i,0,0,0,0,0,0,0]) for i in [1,2,3]], np.random.default_rng(42))['M_recovered'][0] - 600) < 500))
    
    def test_cdks_vuln():
        rng = np.random.default_rng(42)
        pp = cdks_setup(8, 1048583, 3, 10, rng)
        sk, pk = cdks_keygen(pp, rng)
        m = np.array([1000, 0, 0, 0, 0, 0, 0, 0], dtype=np.int64)
        c0, c1 = cdks_encrypt(pp, pk, m, rng)
        mu = cdks_partial_decrypt(sk, c1, pp, rng)
        assert abs(int(poly_mod_q(c0 + mu, 1048583)[0]) - 1000) < 500
    check("cdks_vulnerability", test_cdks_vuln)

    # 15-19: Shamir Tests
    check("shamir_share", lambda: assert_true(len(ClassicShamir.share(42, 3, [1,2,3,4,5], 104729)) == 5))
    check("shamir_reconstruction", lambda: assert_true(ClassicShamir.combine(ClassicShamir.share(42, 3, [1,2,3,4,5], 104729)[:3], 104729) == 42))
    check("exceptional_sequence", lambda: assert_true(RingShamir.check_exceptional_sequence([1,2,3,4,5], 1048583) and not RingShamir.check_exceptional_sequence([1,1,3], 1048583)))
    check("ring_shamir", lambda: assert_true(np.array_equal(RingShamir.combine(RingShamir.share(np.array([10,20,30,40,50,60,70,80]), 2, [1,2,3], 8, 1048583)[:2], 8, 1048583), np.array([10,20,30,40,50,60,70,80]))))
    check("share_resharing", lambda: assert_true(RingShamir.combine(reshare([np.array([10,0,0,0,0,0,0,0]), np.array([20,0,0,0,0,0,0,0]), np.array([30,0,0,0,0,0,0,0])], 2, [1,2,3], 8, 1048583)[:2], 8, 1048583)[0] == 60))

    # 20-21: CDKS-LSS Tests
    check("cdks_lss_decryption", lambda: assert_true(full_cdks_lss_pipeline(cdks_setup(8, 1048583, 3, 10, np.random.default_rng(42)), [np.array([100*i,0,0,0,0,0,0,0]) for i in [1,2,3]], 2, [1,2,3], rng=np.random.default_rng(42))['max_error'] < 500))
    check("threshold_reconstruction", lambda: assert_true(full_cdks_lss_pipeline(cdks_setup(8, 1048583, 3, 10, np.random.default_rng(42)), [np.array([100*i,0,0,0,0,0,0,0]) for i in range(1,6)], 2, [1,2,3,4,5], available_indices=[0, 1], rng=np.random.default_rng(42))['max_error'] < 500))

    # 22-23: FL & Math Validation
    def test_fl():
        cd, td = generate_fl_dataset(3, 50, 10, non_iid=False, random_state=42)
        clients = [FLClient(i, X, y) for i, (X, y) in enumerate(cd)]
        h = run_fedavg(clients, td, 10, n_rounds=5, rng=np.random.default_rng(42))
        assert h['accuracy'][-1] > 0.5
    check("fedavg_convergence", test_fl)

    def test_math_val():
        pp = cdks_setup(8, 1048583, 3, 10, np.random.default_rng(42))
        res = full_cdks_lss_pipeline(pp, [np.array([10*i,0,0,0,0,0,0,0]) for i in [1,2,3]], 2, [1,2,3], rng=np.random.default_rng(42))
        assert res['max_error'] < 500
    check("mathematical_validation", test_math_val)

    print("=" * 60)
    print(f"TEST RESULTS: {passed_tests} PASSED, {failed_tests} FAILED")
    print("=" * 60)

def assert_true(cond):
    assert cond

run_all_unit_tests()


## 14. Experiment 1: Plain FedAvg Baseline

Evaluates standard Federated Averaging across $N=5$ clients with non-IID data distributions over 20 communication rounds.


In [ ]:
# ============================================================================
# EXPERIMENT 1: PLAIN FedAvg BASELINE
# ============================================================================

def run_experiment_1():
    print("\n" + "#" * 60)
    print("# EXPERIMENT 1: Plain FedAvg Baseline")
    print("#" * 60)
    rng = np.random.default_rng(RANDOM_SEED)
    client_data, test_data = generate_fl_dataset(
        n_clients=NUM_CLIENTS, n_samples_per_client=NUM_SAMPLES_PER_CLIENT,
        n_features=NUM_FEATURES, non_iid=True, random_state=RANDOM_SEED
    )
    clients = [FLClient(i, X, y) for i, (X, y) in enumerate(client_data)]
    history = run_fedavg(
        clients, test_data, NUM_FEATURES, n_rounds=FL_ROUNDS,
        learning_rate=LEARNING_RATE, local_epochs=LOCAL_EPOCHS, rng=rng
    )
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    rounds = range(1, FL_ROUNDS + 1)
    axes[0].plot(rounds, history['accuracy'], 'b-o', markersize=4)
    axes[0].set_xlabel('Communication Round')
    axes[0].set_ylabel('Test Accuracy')
    axes[0].set_title('FedAvg — Test Accuracy')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(rounds, history['loss'], 'r-o', markersize=4)
    axes[1].set_xlabel('Communication Round')
    axes[1].set_ylabel('Test Loss')
    axes[1].set_title('FedAvg — Test Loss')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'experiment_fedavg.png'), dpi=150)
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)

    print(f"FedAvg Final Accuracy: {history['accuracy'][-1]:.4f}, Final Loss: {history['loss'][-1]:.4f}")
    return history

exp1_hist = run_experiment_1()


## 15. Experiment 2: CDKS Encrypted FL + Vulnerability Demonstration

Runs FL with standard CDKS encryption and proves that the central server can recover individual model weights via:
$$m_i \approx c_{i,0} + \mu_i$$


In [ ]:
# ============================================================================
# EXPERIMENT 2: CDKS FL + VULNERABILITY DEMONSTRATION
# ============================================================================

def run_experiment_2():
    print("\n" + "#" * 60)
    print("# EXPERIMENT 2: CDKS Encrypted FL + Vulnerability")
    print("#" * 60)
    rng = np.random.default_rng(RANDOM_SEED)
    pp = cdks_setup(RING_N, RING_Q, ERROR_BOUND, SMUDGE_BOUND, rng)
    client_data, test_data = generate_fl_dataset(
        n_clients=NUM_CLIENTS, n_samples_per_client=NUM_SAMPLES_PER_CLIENT,
        n_features=NUM_FEATURES, non_iid=True, random_state=RANDOM_SEED
    )
    clients = [FLClient(i, X, y) for i, (X, y) in enumerate(client_data)]
    history = run_cdks_fl(
        clients, test_data, pp, NUM_FEATURES, n_rounds=FL_ROUNDS,
        learning_rate=LEARNING_RATE, local_epochs=LOCAL_EPOCHS,
        weight_scale=WEIGHT_SCALE, rng=rng
    )

    # Vulnerability Proof on Toy Plaintexts
    print("\n--- CDKS Vulnerability Attack Proof ---")
    rng_demo = np.random.default_rng(99)
    pp_demo = cdks_setup(8, RING_Q, ERROR_BOUND, SMUDGE_BOUND, rng_demo)
    demo_plaintexts = [np.array([100*i, 200*i, 0, 0, 0, 0, 0, 0], dtype=np.int64) for i in [1, 2, 3]]
    demo_res = full_cdks_pipeline(pp_demo, demo_plaintexts, rng_demo)
    for i in range(3):
        attack = attack_cdks_partial_decryption(demo_res['ciphertexts'][i][0], demo_res['partial_decryptions'][i], demo_plaintexts[i], RING_Q)
        print(f"  Client {i+1}: True={demo_plaintexts[i][:2]}, Recovered by Server={attack['recovered'][:2]} (Error={attack['max_error']})")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    rounds = range(1, FL_ROUNDS + 1)
    axes[0].plot(rounds, history['accuracy'], 'g-o', markersize=4, label='CDKS FL')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('CDKS FL Accuracy')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(rounds, history['loss'], 'r-o', markersize=4, label='CDKS FL')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('CDKS FL Loss')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'experiment_cdks.png'), dpi=150)
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)

    print(f"CDKS FL Final Accuracy: {history['accuracy'][-1]:.4f}")
    return history

exp2_hist = run_experiment_2()


## 16. Experiment 3: CDKS-LSS Privacy-Preserving FL (Proposed Method)

Evaluates the proposed CDKS-LSS framework side-by-side with FedAvg on identical non-IID datasets. Demonstrates that secret sharing partial decryptions preserves accuracy while guaranteeing individual privacy.


In [ ]:
# ============================================================================
# EXPERIMENT 3: PROPOSED CDKS-LSS FL vs FedAvg
# ============================================================================

def run_experiment_3():
    print("\n" + "#" * 60)
    print("# EXPERIMENT 3: CDKS-LSS Privacy-Preserving FL")
    print("#" * 60)
    alphas = get_evaluation_points(NUM_CLIENTS)
    client_data, test_data = generate_fl_dataset(
        n_clients=NUM_CLIENTS, n_samples_per_client=NUM_SAMPLES_PER_CLIENT,
        n_features=NUM_FEATURES, non_iid=True, random_state=RANDOM_SEED
    )

    # 1. FedAvg
    rng1 = np.random.default_rng(RANDOM_SEED)
    clients1 = [FLClient(i, X, y) for i, (X, y) in enumerate(client_data)]
    hist_fedavg = run_fedavg(clients1, test_data, NUM_FEATURES, FL_ROUNDS, LEARNING_RATE, LOCAL_EPOCHS, rng1)

    # 2. CDKS-LSS
    rng2 = np.random.default_rng(RANDOM_SEED)
    pp = cdks_setup(RING_N, RING_Q, ERROR_BOUND, SMUDGE_BOUND, rng2)
    clients2 = [FLClient(i, X, y) for i, (X, y) in enumerate(client_data)]
    hist_lss = run_cdks_lss_fl(
        clients2, test_data, pp, NUM_FEATURES, THRESHOLD, alphas,
        FL_ROUNDS, LEARNING_RATE, LOCAL_EPOCHS, WEIGHT_SCALE, rng=rng2
    )

    # Comparison Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    rounds = range(1, FL_ROUNDS + 1)
    axes[0].plot(rounds, hist_fedavg['accuracy'], 'b-o', markersize=4, label='FedAvg (Unencrypted)')
    axes[0].plot(rounds, hist_lss['accuracy'], 'r-s', markersize=4, label='CDKS-LSS (Proposed)')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Test Accuracy')
    axes[0].set_title('Accuracy: FedAvg vs CDKS-LSS')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(rounds, hist_fedavg['loss'], 'b-o', markersize=4, label='FedAvg (Unencrypted)')
    axes[1].plot(rounds, hist_lss['loss'], 'r-s', markersize=4, label='CDKS-LSS (Proposed)')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Test Loss')
    axes[1].set_title('Loss: FedAvg vs CDKS-LSS')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'experiment_cdks_lss.png'), dpi=150)
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)

    print(f"FedAvg Final Accuracy:   {hist_fedavg['accuracy'][-1]:.4f}")
    print(f"CDKS-LSS Final Accuracy: {hist_lss['accuracy'][-1]:.4f}")
    return {'fedavg': hist_fedavg, 'cdks_lss': hist_lss}

exp3_hist = run_experiment_3()


## 17. Experiment 4: xMK-CKKS Baseline Comparison

Compares FedAvg, xMK-CKKS, and CDKS-LSS under identical conditions.


In [ ]:
# ============================================================================
# EXPERIMENT 4: THREE-WAY COMPARISON (FedAvg vs xMK-CKKS vs CDKS-LSS)
# ============================================================================

def run_experiment_4():
    print("\n" + "#" * 60)
    print("# EXPERIMENT 4: Three-Way Comparison")
    print("#" * 60)
    alphas = get_evaluation_points(NUM_CLIENTS)
    client_data, test_data = generate_fl_dataset(
        n_clients=NUM_CLIENTS, n_samples_per_client=NUM_SAMPLES_PER_CLIENT,
        n_features=NUM_FEATURES, non_iid=True, random_state=RANDOM_SEED
    )

    # 1. FedAvg
    rng1 = np.random.default_rng(RANDOM_SEED)
    clients1 = [FLClient(i, X, y) for i, (X, y) in enumerate(client_data)]
    h_fedavg = run_fedavg(clients1, test_data, NUM_FEATURES, FL_ROUNDS, LEARNING_RATE, LOCAL_EPOCHS, rng1)

    # 2. xMK-CKKS
    rng2 = np.random.default_rng(RANDOM_SEED)
    pp2 = cdks_setup(RING_N, RING_Q, ERROR_BOUND, SMUDGE_BOUND, rng2)
    clients2 = [FLClient(i, X, y) for i, (X, y) in enumerate(client_data)]
    h_xmk = run_xmk_ckks_fl(clients2, test_data, pp2, NUM_FEATURES, FL_ROUNDS, LEARNING_RATE, LOCAL_EPOCHS, WEIGHT_SCALE, rng2)

    # 3. CDKS-LSS
    rng3 = np.random.default_rng(RANDOM_SEED)
    pp3 = cdks_setup(RING_N, RING_Q, ERROR_BOUND, SMUDGE_BOUND, rng3)
    clients3 = [FLClient(i, X, y) for i, (X, y) in enumerate(client_data)]
    h_lss = run_cdks_lss_fl(clients3, test_data, pp3, NUM_FEATURES, THRESHOLD, alphas, FL_ROUNDS, LEARNING_RATE, LOCAL_EPOCHS, WEIGHT_SCALE, rng=rng3)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    rounds = range(1, FL_ROUNDS + 1)
    for ax_i, k, title in [(0, 'accuracy', 'Accuracy'), (1, 'loss', 'Loss')]:
        axes[ax_i].plot(rounds, h_fedavg[k], 'b-o', ms=3, label='FedAvg')
        axes[ax_i].plot(rounds, h_xmk[k], 'g-^', ms=3, label='xMK-CKKS')
        axes[ax_i].plot(rounds, h_lss[k], 'r-s', ms=3, label='CDKS-LSS (Proposed)')
        axes[ax_i].set_xlabel('Round')
        axes[ax_i].set_ylabel(f'Test {title}')
        axes[ax_i].set_title(f'Three-Way {title} Comparison')
        axes[ax_i].legend()
        axes[ax_i].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'experiment_xmk_ckks.png'), dpi=150)
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)

    print(f"Final Accuracies -> FedAvg: {h_fedavg['accuracy'][-1]:.4f}, xMK-CKKS: {h_xmk['accuracy'][-1]:.4f}, CDKS-LSS: {h_lss['accuracy'][-1]:.4f}")
    return {'fedavg': h_fedavg, 'xmk': h_xmk, 'cdks_lss': h_lss}

exp4_hist = run_experiment_4()


## 18. Experiment 5: Security Attacks Comprehensive Evaluation

Demonstrates the security properties of Plain FL, CDKS, and CDKS-LSS against an honest-but-curious server attempting to recover individual client updates.


In [ ]:
# ============================================================================
# EXPERIMENT 5: SECURITY EVALUATION
# ============================================================================

def run_experiment_5():
    run_all_attacks()
    print("\nSecurity Summary:")
    print(f"  {'Method':<12} | {'Individual Privacy':<22} | {'Server View':<30}")
    print("  " + "-" * 70)
    print(f"  {'Plain FL':<12} | {'NO (Completely Exposed)':<22} | {'Raw local weights w_i':<30}")
    print(f"  {'CDKS':<12} | {'NO (Leaked via mu_i)':<22} | {'m_i approx c_i0 + mu_i':<30}")
    print(f"  {'CDKS-LSS':<12} | {'YES (Provably Protected)':<22} | {'Aggregated shares sum f_i(alpha_j)':<30}")

run_experiment_5()


## 19. Experiment 6: Client Dropout Tolerance Evaluation

Demonstrates the threshold property of CDKS-LSS ($N=10, t=6$) against xMK-CKKS when clients drop out before the decryption phase.


In [ ]:
# ============================================================================
# EXPERIMENT 6: CLIENT DROPOUT TOLERANCE
# ============================================================================

def run_experiment_6():
    print("\n" + "#" * 60)
    print("# EXPERIMENT 6: Client Dropout Tolerance (N=10, t=6)")
    print("#" * 60)
    n_ring, q, N, t = 8, RING_Q, 10, 6
    alphas = list(range(1, N + 1))
    plaintexts = []
    for i in range(N):
        m = np.zeros(n_ring, dtype=np.int64)
        m[0] = (i + 1) * 100
        plaintexts.append(m)

    cdks_results = []
    print("\n--- Testing CDKS-LSS with Varying Available Clients ---")
    for n_avail in range(N, t - 1, -1):
        avail = list(range(n_avail))
        rng = np.random.default_rng(42)
        pp = cdks_setup(n_ring, q, ERROR_BOUND, SMUDGE_BOUND, rng)
        res = full_cdks_lss_pipeline(pp, plaintexts, t, alphas, available_indices=avail, rng=rng)
        succ = res['max_error'] < 1000
        cdks_results.append({'avail': n_avail, 'success': succ, 'error': res['max_error']})
        print(f"  CDKS-LSS: {n_avail:2d}/{N} online ({N-n_avail} dropped) -> Success: {succ} (Error={res['max_error']})")

    xmk_results = []
    print("\n--- Testing xMK-CKKS with Varying Available Clients ---")
    for n_avail in [N, N - 1, N - 2]:
        avail = list(range(n_avail))
        rng = np.random.default_rng(42)
        pp = cdks_setup(n_ring, q, ERROR_BOUND, SMUDGE_BOUND, rng)
        res = full_xmk_ckks_pipeline(pp, plaintexts, available_indices=avail, rng=rng)
        succ = res['max_error'] < 1000
        xmk_results.append({'avail': n_avail, 'success': succ, 'error': res['max_error']})
        print(f"  xMK-CKKS: {n_avail:2d}/{N} online ({N-n_avail} dropped) -> Success: {succ} (Error={res['max_error']})")

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    c_x = [r['avail'] - 0.2 for r in cdks_results]
    c_y = [1 if r['success'] else 0 for r in cdks_results]
    ax.bar(c_x, c_y, width=0.4, color='#2ecc71', alpha=0.85, label='CDKS-LSS (Proposed)')

    x_x = [r['avail'] + 0.2 for r in xmk_results]
    x_y = [1 if r['success'] else 0 for r in xmk_results]
    ax.bar(x_x, x_y, width=0.4, color='#e74c3c', alpha=0.85, label='xMK-CKKS (Baseline)')

    ax.axvline(x=t, color='black', linestyle='--', label=f'Threshold t={t}')
    ax.set_xlabel('Available Clients during Decryption Phase')
    ax.set_ylabel('Decryption Success (1=Pass, 0=Fail)')
    ax.set_title('Client Dropout Resilience: CDKS-LSS vs xMK-CKKS')
    ax.set_xticks(range(t, N + 1))
    ax.set_yticks([0, 1])
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'experiment_dropout.png'), dpi=150)
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)

run_experiment_6()


## 20. Experiment 7: Communication Overhead Analysis

Analyzes the theoretical and concrete communication overheads (Table 2 in paper) across FedAvg, xMK-CKKS, and CDKS-LSS.


In [ ]:
# ============================================================================
# EXPERIMENT 7: COMMUNICATION OVERHEAD
# ============================================================================

def run_experiment_7():
    print("\n" + "#" * 60)
    print("# EXPERIMENT 7: Communication Overhead Analysis")
    print("#" * 60)
    num_weight_polys = max(1, math.ceil(NUM_FEATURES / RING_N))
    overhead = communication_overhead(RING_N, RING_Q, NUM_CLIENTS, THRESHOLD, num_weight_polys)
    print(format_overhead_table(overhead))

    methods = list(overhead.keys())
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    colors = ['#3498db', '#e67e22', '#2ecc71']

    # 1. Total Bytes
    tot_kb = [overhead[m]['total_bytes'] / 1024 for m in methods]
    axes[0].bar(methods, tot_kb, color=colors)
    axes[0].set_ylabel('Total Communication (KB)')
    axes[0].set_title('Total Communication per Round')
    axes[0].grid(True, alpha=0.3, axis='y')

    # 2. Total Messages
    tot_msgs = [overhead[m]['total_msgs'] for m in methods]
    axes[1].bar(methods, tot_msgs, color=colors)
    axes[1].set_ylabel('Messages per Round')
    axes[1].set_title('Message Complexity')
    axes[1].grid(True, alpha=0.3, axis='y')

    # 3. Client-to-Client Bytes
    c2c_kb = [overhead[m]['client_to_client_bytes'] / 1024 for m in methods]
    axes[2].bar(methods, c2c_kb, color=colors)
    axes[2].set_ylabel('Client<->Client Traffic (KB)')
    axes[2].set_title('P2P Secret-Sharing Traffic')
    axes[2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'experiment_overhead.png'), dpi=150)
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)

run_experiment_7()


## 21. Master Benchmark Runner

A unified orchestrator function to run any specific component demo or full experiment on demand.


In [ ]:
# ============================================================================
# MASTER EXPERIMENT RUNNER
# ============================================================================

def run_all():
    print("=" * 70)
    print("RUNNING COMPLETE CDKS-LSS SUITE (ALL EXPERIMENTS & DEMOS)")
    print("=" * 70)
    t_start = time.time()
    demo_rlwe()
    demo_cdks()
    demo_shamir_all()
    demo_cdks_lss()
    demo_xmk_ckks()
    run_all_unit_tests()
    run_experiment_1()
    run_experiment_2()
    run_experiment_3()
    run_experiment_4()
    run_experiment_5()
    run_experiment_6()
    run_experiment_7()
    print(f"\nAll demonstrations and experiments finished in {time.time() - t_start:.2f}s.")

print("Master benchmark runner ready. Call run_all() to execute the complete pipeline.")


## 22. Thesis & Viva Defense Guide

### 9 Critical Viva Questions & Answers

#### Q1: Why is plain Federated Learning (FedAvg) insufficient for privacy?
> **Answer:** In FedAvg, clients transmit raw weight vectors $w_i$ or gradients $\nabla L(w_i)$ directly to the central aggregator. Extensive research (e.g., Zhu et al., Deep Leakage from Gradients) demonstrates that an untrusted server can invert these gradients to faithfully reconstruct private client training data.

#### Q2: What is CDKS Multi-Key Homomorphic Encryption, and why was it chosen?
> **Answer:** CDKS is an RLWE-based multi-key homomorphic encryption scheme. Unlike single-key HE, it allows each client $i$ to encrypt updates under their **own independent key pair** $(sk_i, pk_i)$ using a shared public polynomial $a$. The server can homomorphically add ciphertexts into a joint multi-key ciphertext $ct_{\text{add}} = (\sum c_{i,0}, c_{1,1}, \dots, c_{N,1})$ without requiring a trusted key generation authority.

#### Q3: What is the fundamental vulnerability in standard CDKS?
> **Answer:** During the collaborative decryption phase of standard CDKS, each client sends their partial decryption value $\mu_i = c_{i,1} \cdot s_i + e_i^*$ to the server. Because $c_{i,0} = v_i b_i + m_i + e_{i,0}$ and $b_i = -a s_i + e_i$, the term $v_i a s_i$ cancels when adding $c_{i,0} + \mu_i$:
> $$c_{i,0} + \mu_i = m_i + (v_i e_i + e_{i,0} + e_{i,1} s_i + e_i^*) \approx m_i$$
> The server can recover each client's individual plaintext model update $m_i$, completely defeating the purpose of homomorphic encryption!

#### Q4: How does Shamir Linear Secret Sharing (LSS) over $R_q$ solve this?
> **Answer:** Instead of sending $\mu_i$ directly to the server, client $i$ constructs a random sharing polynomial $f_i(X)$ over the ring $R_q$ of degree $t-1$ such that $f_i(0) = \mu_i$. Clients exchange shares $f_i(\alpha_j)$ peer-to-peer. Each client $j$ computes their aggregated share $\tilde{s}_j = \sum_{i=1}^N f_i(\alpha_j)$ and submits only $\tilde{s}_j$ to the server.

#### Q5: Why does the server learn only the sum $\sum \mu_i$ and not individual $\mu_i$?
> **Answer:** The aggregated shares are evaluations of the sum polynomial $F(X) = \sum_{i=1}^N f_i(X)$. Evaluating $F(0)$ via Lagrange interpolation yields $\sum f_i(0) = \sum \mu_i$. The individual polynomials $f_i(X)$ and their constant terms $\mu_i$ remain information-theoretically protected.

#### Q6: What is an "Exceptional Sequence" in the polynomial ring $R_q$?
> **Answer:** By Definition 2 in the paper, evaluation points $(\alpha_1, \dots, \alpha_N)$ form an exceptional sequence in $R_q$ if $(\alpha_i - \alpha_j)$ is an invertible unit in $R_q$ for all $i \ne j$. For integer evaluation points embedded as constant polynomials, $\gcd(\alpha_i - \alpha_j, q) = 1$ ensures invertibility, which holds because $q$ is prime and $|\alpha_i - \alpha_j| < q$.

#### Q7: Why is CDKS-LSS superior to xMK-CKKS regarding client dropout?
> **Answer:** In xMK-CKKS, all participants encrypt under a unified public key $\tilde{b} = \sum b_i$. Decryption requires partial decryptions from **all $N$ participants**. If a single client drops out, the system cannot decrypt. In CDKS-LSS, $(t, N)$-threshold sharing requires only $t$ aggregated shares. Up to $N - t$ clients can drop out without impeding model update recovery.

#### Q8: What is the communication trade-off of CDKS-LSS?
> **Answer:** CDKS-LSS introduces client-to-client (peer-to-peer) communication for exchanging secret shares: $N(N-1)$ messages of size $n \lceil \log_2 q \rceil$ bits. This extra communication is the deliberate trade-off required to eliminate partial decryption leakage and provide threshold dropout tolerance.

#### Q9: What are the primary assumptions and deviations in this educational implementation?
> **Answer:**
> 1. Parameters: $n=64, q=1048583$ (for rapid execution and coefficient inspection; production requires $n \ge 4096, q \sim 2^{100+}$).
> 2. Noise: Bounded uniform noise approximation rather than continuous discrete Gaussian sampling.
> 3. Ring Multiplication: Naive negacyclic convolution ($O(n^2)$) rather than Number Theoretic Transform ($O(n \log n)$).
> 4. Weight Quantization: Fixed-point scaling via integer rounding.
